# Example 4: ORF Finding in a Real Plasmid — pUC19


Fetches the complete sequence of pUC19 (GenBank `L09137`, 2,686 bp) — a small, widely used
cloning vector — and runs the same six-frame scan across the whole plasmid,
with no coordinates supplied in advance.


pUC19's reference map places both *bla* (AmpR, `complement(1626..2486)`,
861 bp → 286 aa) and the *lacZα* fragment (`complement(146..469)`, 324 bp → 107 aa) on the
**reverse strand** — a genuine test that the scan's reverse-strand half does real work.

In [2]:
# Fetch pUC19 and scan all six reading frames for its own annotated genes
import os
from Bio import Entrez, SeqIO

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

Entrez.email = "you@example.com"
handle = Entrez.efetch(db="nucleotide", id="L09137", rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()
SeqIO.write(record, "data/L09137_pUC19.fasta", "fasta")
print(f"pUC19 length: {len(record.seq)} bp")
print("Treated as linear below; pUC19 is actually circular, so an ORF spanning")
print("the origin (position 2686/1) would be missed — a known limitation of a simple linear scan.")

def find_orfs(seq, min_aa=80):
    """Same scanning function as Example 2 — three frames per strand."""
    orfs = []
    for frame in range(3):
        # Remove incomplete codons at the end of this reading frame
        coding_seq = seq[frame:]
        coding_seq = coding_seq[:len(coding_seq) // 3 * 3]
        
        trans = str(coding_seq.translate(table=1, to_stop=False))
        start = None
        for i, aa in enumerate(trans):
            if aa == "M" and start is None:
                start = i
            if aa == "*" and start is not None:
                length = i - start
                if length >= min_aa:
                    nt_start = frame + start * 3
                    nt_end = frame + (i + 1) * 3
                    orfs.append((frame, nt_start, nt_end, length))
                start = None
    return orfs

forward_orfs = find_orfs(record.seq)
revcomp = record.seq.reverse_complement()
reverse_orfs = find_orfs(revcomp)

all_orfs = (
    [("forward", f, s, e, l) for f, s, e, l in forward_orfs] +
    [("reverse", f, s, e, l) for f, s, e, l in reverse_orfs]
)
all_orfs.sort(key=lambda x: -x[4])

print(f"\nORFs found >= 80 aa: {len(all_orfs)}")
for strand, frame, s, e, length in all_orfs:
    print(f"  {strand} frame {frame+1}: {s}-{e} nt, {length} aa")

with open("results/example4_puc19_orfs.csv", "w") as f:
    f.write("strand,frame,nt_start,nt_end,aa_length\n")
    for strand, frame, s, e, length in all_orfs:
        f.write(f"{strand},{frame+1},{s},{e},{length}\n")
print("\nSaved: data/L09137_pUC19.fasta, results/example4_puc19_orfs.csv")

pUC19 length: 2686 bp
Treated as linear below; pUC19 is actually circular, so an ORF spanning
the origin (position 2686/1) would be missed — a known limitation of a simple linear scan.

ORFs found >= 80 aa: 4
  reverse frame 3: 200-1061 nt, 286 aa
  forward frame 3: 185-542 nt, 118 aa
  reverse frame 1: 2217-2541 nt, 107 aa
  forward frame 1: 1755-2022 nt, 88 aa

Saved: data/L09137_pUC19.fasta, results/example4_puc19_orfs.csv
